In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# 1. Update Linux system tools
!apt-get update && apt-get install -y build-essential git

# 2. Upgrade CMake to version 3.24+ (Required for CUDA detection)
!pip install cmake --upgrade

# 3. Install Python dependencies for llama.cpp conversion scripts
!pip install -r https://raw.githubusercontent.com/ggerganov/llama.cpp/master/requirements.txt
!pip install huggingface_hub datasets

# 4. CRITICAL FIX: Link the CUDA driver
# Kaggle has libcuda.so.1 but lacks the libcuda.so symlink required for linking.
!ln -sf $(find /usr -name libcuda.so.1 | head -n 1) /usr/lib/x86_64-linux-gnu/libcuda.so

print("Environment setup complete. CUDA driver linked.")

In [ ]:
# 1. Clone the repository
!git clone https://github.com/ggerganov/llama.cpp

# 2. Configure the build with CUDA (GPU) support
# We use 'python -m cmake' to ensure we use the upgraded version from Cell 1
!python -m cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=1

# 3. Compile the binaries (Release mode for speed)
!python -m cmake --build llama.cpp/build --config Release -j $(nproc)

# 4. Copy the resulting binaries to the main folder for easy access
!cp llama.cpp/build/bin/* llama.cpp/

print("Build complete. Llama.cpp binaries are ready.")

In [ ]:
from huggingface_hub import snapshot_download
from datasets import load_dataset
import os

# 1. Download the base model (Microsoft Phi-2)
print("Downloading Phi-2...")
snapshot_download(repo_id="microsoft/phi-2", local_dir="phi-2-hf", local_dir_use_symlinks=False)

# 2. Download Calibration Data (WikiText-2)
# We need this to teach the quantizer which weights are important.
print("Downloading calibration data...")
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")

# 3. Save the dataset as a plain text file
with open("wiki.train.raw", "w", encoding="utf-8") as f:
    for line in dataset["text"]:
        f.write(line)

print("Downloads finished. Files ready.")

In [ ]:
# Convert the 'phi-2-hf' folder to a GGUF file (FP16 precision)
!python llama.cpp/convert_hf_to_gguf.py phi-2-hf --outfile phi-2-f16.gguf --outtype f16

In [ ]:
# Calculate weight importance using the GPU (-ngl 99)
# -m: Input model
# -f: Calibration text file
# -o: Output matrix file
!./llama.cpp/llama-imatrix -m phi-2-f16.gguf -f wiki.train.raw -o imatrix.dat -ngl 99

In [ ]:
# Create the quantized model using the matrix
!./llama.cpp/llama-quantize --imatrix imatrix.dat phi-2-f16.gguf phi-2-IQ4_XS.gguf IQ4_XS

In [ ]:
from huggingface_hub import HfApi, login

# --- CONFIGURATION ---
hf_token = "x"  # Your WRITE token
model_name = "phi-2-GGUF-IQ4_XS"                     # Name of the repo to create
username = "eziyo"                                   # Your username
# ---------------------

login(token=hf_token)
api = HfApi()
repo_id = f"{username}/{model_name}"

# 1. Create the repository
print(f"Creating repo: {repo_id}")
api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)

# 2. Define files to upload (Model + Matrix + Readme)
# We create a simple README dynamically
with open("README.md", "w") as f:
    f.write(f"---\nlicense: mit\nbase_model: microsoft/phi-2\ntags:\n  - quantization\n  - gguf\n  - iq4_xs\n  - imatrix\n---\n# {model_name}\n\nQuantized using llama.cpp with IQ4_XS.")

files = ["phi-2-IQ4_XS.gguf", "imatrix.dat", "README.md"]

# 3. Upload loop
for file in files:
    print(f"Uploading {file}...")
    try:
        api.upload_file(
            path_or_fileobj=file,
            path_in_repo=file,
            repo_id=repo_id,
            repo_type="model"
        )
        print(f"✅ Uploaded {file}")
    except Exception as e:
        print(f"❌ Failed to upload {file}: {e}")

print("All tasks finished successfully.")